# Multimodal Emotion Detection - Fusion of ResNet50 and DistilBERT

## Imports

In [1]:


import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import pandas as pd
import numpy as np
from transformers import DistilBertTokenizer, DistilBertModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter


C:\Users\USER\PycharmProjects\DSGP15_Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Tokenizer and Model

In [2]:
# Cell 2: Load tokenizer and model
tokenizer = DistilBertTokenizer.from_pretrained("../DistilBERT/saved_emotion_bert")
bert = DistilBertModel.from_pretrained("../DistilBERT/saved_emotion_bert")
bert.eval()


DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSdpaAttention(
          (dropout): Dropout(p=0.1, inplace=False)
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=True)
          (lin2): L

## Function to extract text features

In [3]:

def extract_text_features(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=64
    )

    with torch.no_grad():
        outputs = bert(**inputs)

    # CLS token embedding (1, 768)
    return outputs.last_hidden_state[:, 0, :]


## Test the function

In [4]:
text = "The child looks very happy and excited"
features = extract_text_features(text)
print("Text feature shape:", features.shape)


Text feature shape: torch.Size([1, 768])


## Fusion model definition

In [ ]:

import torch.nn as nn

class MultimodalEmotionClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        # 2048 from ResNet + 768 from BERT
        self.fc = nn.Sequential(
            nn.Linear(2048 + 768, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 2)  # happy / sad
        )

    def forward(self, image_feat, text_feat):
        # concatenate features along dim=1
        fused = torch.cat((image_feat, text_feat), dim=1)
        return self.fc(fused)


## Instantiate fusion model

In [ ]:

fusion_model = MultimodalEmotionClassifier().to(device)
fusion_model.eval()  # we’ll test first, train later


## Function to predict emotion from image + text

In [ ]:

def predict_emotion_multimodal(image_path, text):
    # Extract image features
    from PIL import Image
    image = Image.open(image_path).convert("RGB")
    image = val_test_transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        img_feat = resnet(image)

    # Extract text features
    txt_feat = extract_text_features(text).to(device)

    # Fusion model prediction
    with torch.no_grad():
        logits = fusion_model(img_feat, txt_feat)
    pred = torch.argmax(logits, dim=1).item()

    return ["sad", "happy"][pred]


## Test prediction

In [ ]:

test_image = "../Images/Emotion/test/happy/child1.jpg"
test_text = "The child looks very happy and excited"

prediction = predict_emotion_multimodal(test_image, test_text)
print("Predicted emotion:", prediction)
